# Разбор Feature Separability `O/B` В Coarse Source

Цель:
- Проверить, separable ли true `O` и true `B` по текущему coarse feature contract.
- Посмотреть, как current coarse artifact ведет себя на train-time `O/B` boundary source.
- Понять, это feature-boundary problem или downstream domain-shift problem.

In [1]:
# Setup: repo root, sys.path и display-настройки.
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
import seaborn as sns
from IPython.display import display

def find_repo_root(start_path: Path) -> Path:
    current_path = start_path.resolve()
    while current_path != current_path.parent:
        if (current_path / 'src').exists() and (current_path / '.git').exists():
            return current_path
        current_path = current_path.parent
    raise RuntimeError('Could not locate repository root.')

REPO_ROOT = find_repo_root(Path.cwd())
SRC_ROOT = REPO_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

sns.set_theme(style='whitegrid', context='talk')
pd.set_option('display.max_columns', 140)
pd.set_option('display.width', 160)


In [2]:
# Импортируем separability helpers.
from exohost.reporting.archive_research.coarse_ob_feature_separability_review import (
    CoarseOBFeatureSeparabilityConfig,
    build_boundary_feature_physics_frame,
    build_boundary_membership_summary_frame,
    build_boundary_predicted_class_summary_frame,
    build_boundary_probability_summary_frame,
    build_boundary_true_class_summary_frame,
    build_univariate_separability_auc_frame,
    load_coarse_ob_feature_separability_review_bundle_from_env,
)


## План

1. Загружаем train-time hot `O/B` boundary source.
2. Смотрим, сколько true `O` и true `B` входит в boundary.
3. Проверяем, как current coarse artifact предсказывает этот source.
4. Сравниваем `P(O)` и `P(B)` на true `O` и true `B`.
5. Смотрим class-level physics summary.
6. Считаем univariate separability по признакам.
7. Считаем permutation importance текущего coarse artifact.
8. В конце фиксируем, это model problem или downstream domain shift.

In [3]:
# Конфигурация notebook.
COARSE_MODEL_RUN_DIR = REPO_ROOT / 'artifacts/models/gaia_id_coarse_classification__hist_gradient_boosting__2026_03_28_215003_509969'
REVIEW_CONFIG = CoarseOBFeatureSeparabilityConfig(hot_teff_min_k=10000.0, permutation_n_repeats=10)
SOURCE_LIMIT = None
DOTENV_PATH = REPO_ROOT / '.env'

if not COARSE_MODEL_RUN_DIR.exists():
    raise FileNotFoundError(COARSE_MODEL_RUN_DIR)


In [4]:
# Загружаем train-time O/B separability bundle.
bundle = load_coarse_ob_feature_separability_review_bundle_from_env(
    coarse_model_run_dir=COARSE_MODEL_RUN_DIR,
    config=REVIEW_CONFIG,
    source_limit=SOURCE_LIMIT,
    dotenv_path=str(DOTENV_PATH),
)

display(build_boundary_membership_summary_frame(bundle))


,hot_teff_min_k,n_rows_source,n_rows_boundary,n_rows_scored
0,10000.0,32986,6000,6000


In [5]:
# Сколько true O/B входит в boundary source и что предсказывает current coarse artifact.
display(build_boundary_true_class_summary_frame(bundle.boundary_df))
display(build_boundary_predicted_class_summary_frame(bundle.scored_boundary_df))


,true_spectral_class,n_rows,share
0,B,3000,0.5
1,O,3000,0.5


,coarse_predicted_label,n_rows,share
0,O,3001,0.500167
1,B,2995,0.499167
2,A,4,0.000667


In [6]:
# Что происходит с полными P(O) и P(B) на true O/B.
display(build_boundary_probability_summary_frame(bundle.scored_boundary_df))


,true_spectral_class,n_rows,median_probability__O,median_probability__B,mean_probability__O,mean_probability__B
0,B,3000,0.000017,0.999842,0.000378,0.997463
1,O,3000,0.999845,0.000018,0.999668,0.000189


In [7]:
# Class-level physics summary на train-time boundary source.
display(build_boundary_feature_physics_frame(bundle.boundary_df))


,true_spectral_class,n_rows,median_teff_gspphot,median_logg_gspphot,median_mh_gspphot,median_bp_rp,median_parallax,median_parallax_over_error,median_ruwe,median_radius_feature
0,B,3000,12355.914,3.77930,-0.46645,1.258621,0.256354,7.635526,1.010951,3.890203
1,O,3000,34604.617,3.89625,0.01860,3.385037,0.138732,4.884386,1.024742,8.796550


In [8]:
# Single-feature separability на train-time boundary source.
display(build_univariate_separability_auc_frame(bundle.boundary_df))


,feature_name,n_rows_used,auc_ovr_o,separability_auc,higher_value_class,median_o,median_b,median_gap
0,teff_gspphot,6000,1.000000,1.000000,O,34604.617000,12355.914000,22248.703000
1,radius_feature,6000,0.958801,0.958801,O,8.796550,3.890203,4.906346
2,bp_rp,6000,0.855231,0.855231,O,3.385037,1.258621,2.126416
3,parallax,6000,0.269282,0.730718,B,0.138732,0.256354,-0.117622
4,mh_gspphot,6000,0.709962,0.709962,O,0.018600,-0.466450,0.485050
5,parallax_over_error,6000,0.362243,0.637757,B,4.884386,7.635526,-2.751141
6,logg_gspphot,6000,0.633953,0.633953,O,3.896250,3.779300,0.116950
7,ruwe,6000,0.503267,0.503267,O,1.024742,1.010951,0.013791


In [9]:
# Permutation importance current coarse artifact на train-time boundary source.
display(bundle.permutation_importance_df)


,metric_name,feature_name,importance_mean,importance_std
0,accuracy,teff_gspphot,0.494033,0.007056
1,accuracy,bp_rp,0.002167,0.000558
2,accuracy,radius_feature,0.001933,0.000238
3,accuracy,logg_gspphot,0.001550,0.000211
4,accuracy,mh_gspphot,0.001467,0.000379
5,accuracy,parallax_over_error,0.000133,0.000067
6,accuracy,parallax,0.000067,0.000082
7,accuracy,ruwe,0.000000,0.000000
8,balanced_accuracy,teff_gspphot,0.494033,0.007056
9,balanced_accuracy,bp_rp,0.002167,0.000558


## Вывод

Если train-time `O/B` boundary separable, а current coarse artifact на этом source работает почти идеально,
то observed collapse `O -> B` на downstream hot pass-slice уже не объясняется самой coarse-моделью как таковой.

В таком случае следующий корректный шаг — сравнивать domains:
- train-time coarse reference source
- downstream hot pass-source из `quality_gated/final_decision`

То есть следующий риск — это source mismatch / domain shift, а не отсутствие `O/B` separability в coarse artifact.